# PJM Western Hub Power Model
**Objective:** measure how weather and natural gas prices drive PJM Western Hub on-peak day-ahead power prices, and use that relationship to produce a daily fair-value estimate.

- **Y:** daily on-peak day-ahead (DA) price at Western Hub, $/MWh (weekdays, hours ending 8 to 23 Eastern, excluding NERC holidays)
- **Data type:** daily time series, 2023 to today
- **X:** PJM-city heating and cooling degree days, Henry Hub gas price (previous trading day), day-of-week and month fixed effects
- **Method:** compare benchmark, smoothing, regression, and SARIMAX models on the same out-of-sample period (2026)

**Caveat:** the backtest uses observed weather for the delivery day, i.e. a perfect weather forecast. Real-world error would be somewhat higher.

Run with **Runtime > Run all**. The first run pulls data (a few minutes); later runs read the cached CSV.

In [ ]:
# Cell 1: install (run every new Colab session)
import subprocess, sys, importlib.util
if importlib.util.find_spec("google.colab") is not None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gridstatusio", "xlrd"], check=False)

In [ ]:
# Cell 2: setup and settings
import os, logging, warnings
import numpy as np
import pandas as pd
import requests
import statsmodels.formula.api as smf
from statsmodels.tsa.statespace.sarimax import SARIMAX
import matplotlib.pyplot as plt
from pandas.tseries.holiday import (AbstractHolidayCalendar, Holiday, USMemorialDay,
                                    USLaborDay, USThanksgivingDay, sunday_to_monday)

warnings.filterwarnings("ignore")
logging.getLogger("gridstatusio").setLevel(logging.WARNING)
pd.set_option("display.width", 160)

START = "2023-01-01"
END = (pd.Timestamp.today().normalize() - pd.Timedelta(days=2)).strftime("%Y-%m-%d")
TEST_START = "2026-01-01"          # train: 2023-2025, test: 2026
HUB = "WESTERN HUB"
CACHE = "pjm_daily.csv"

# PJM demand centers (equal-weighted)
CITIES = {
    "philadelphia": (39.95, -75.17), "newark": (40.74, -74.17),
    "baltimore": (39.29, -76.61), "washington": (38.91, -77.04),
    "richmond": (37.54, -77.44), "pittsburgh": (40.44, -79.99),
    "columbus": (39.96, -83.00), "chicago": (41.88, -87.63),
}
print("Data window:", START, "to", END)

In [ ]:
# Cell 3: data functions (prices, weather, gas)
class NERCHolidays(AbstractHolidayCalendar):
    rules = [
        Holiday("New Year", month=1, day=1, observance=sunday_to_monday),
        USMemorialDay,
        Holiday("Independence Day", month=7, day=4, observance=sunday_to_monday),
        USLaborDay,
        USThanksgivingDay,
        Holiday("Christmas", month=12, day=25, observance=sunday_to_monday),
    ]

HOLIDAYS = NERCHolidays().holidays(start="2022-12-01", end="2027-12-31")

def get_client():
    from gridstatusio import GridStatusClient
    try:
        from google.colab import userdata
        key = userdata.get("GRIDSTATUS_API_KEY")
    except Exception:
        key = os.environ.get("GRIDSTATUS_API_KEY")
    return GridStatusClient(api_key=key)

def pull_hub(client, dataset):
    df = client.get_dataset(dataset=dataset, start=START, end=END,
                            filter_column="location", filter_value=HUB)
    return df[["interval_start_utc", "lmp"]]

def daily_blocks(hourly, prefix):
    """Hourly LMP -> daily on-peak and off-peak averages (Eastern time, NERC holidays off-peak)."""
    t = pd.to_datetime(hourly["interval_start_utc"], utc=True).dt.tz_convert("America/New_York")
    d = pd.DataFrame({"t": t.values, "lmp": hourly["lmp"].values})
    d["t"] = pd.to_datetime(d["t"], utc=True).dt.tz_convert("America/New_York")
    d["date"] = d["t"].dt.tz_localize(None).dt.normalize()
    d["onpeak"] = ((d["t"].dt.dayofweek < 5) & d["t"].dt.hour.between(7, 22)
                   & ~d["date"].isin(HOLIDAYS))
    out = d.groupby(["date", "onpeak"])["lmp"].mean().unstack()
    out = out.rename(columns={True: f"{prefix}_on", False: f"{prefix}_off"})
    return out

def get_weather():
    url = "https://archive-api.open-meteo.com/v1/archive"
    frames = []
    for name, (lat, lon) in CITIES.items():
        r = requests.get(url, params={
            "latitude": lat, "longitude": lon, "start_date": START, "end_date": END,
            "daily": "temperature_2m_mean", "temperature_unit": "fahrenheit",
            "timezone": "America/New_York"}, timeout=60)
        js = r.json()["daily"]
        frames.append(pd.Series(js["temperature_2m_mean"], index=pd.to_datetime(js["time"]), name=name))
    temps = pd.concat(frames, axis=1)
    w = pd.DataFrame(index=temps.index)
    w["temp"] = temps.mean(axis=1)
    w["hdd"] = (65 - temps).clip(lower=0).mean(axis=1)
    w["cdd"] = (temps - 65).clip(lower=0).mean(axis=1)
    w.index.name = "date"
    return w

def get_gas():
    """Henry Hub daily spot ($/MMBtu) from EIA. Fallback: upload henry_hub.csv with columns date,gas."""
    try:
        raw = pd.read_excel("https://www.eia.gov/dnav/ng/hist_xls/RNGWHHDd.xls",
                            sheet_name="Data 1", skiprows=2)
        g = raw.iloc[:, :2]
        g.columns = ["date", "gas"]
    except Exception as e:
        print("EIA download failed (", e, "). Reading henry_hub.csv instead.")
        g = pd.read_csv("henry_hub.csv")
    g["date"] = pd.to_datetime(g["date"])
    return g.dropna().sort_values("date")

In [ ]:
# Cell 4: build or load the daily dataset
if os.path.exists(CACHE):
    daily = pd.read_csv(CACHE, parse_dates=["date"])
    print("Loaded cache:", CACHE, daily.shape)
else:
    client = get_client()
    da = daily_blocks(pull_hub(client, "pjm_lmp_day_ahead_hourly"), "da")
    rt = daily_blocks(pull_hub(client, "pjm_lmp_real_time_hourly"), "rt")
    weather = get_weather()
    gas = get_gas()
    daily = da.join(rt, how="left").join(weather, how="left").reset_index()
    # Gas known before the DA market: last Henry Hub price traded strictly before the delivery day
    daily["gas_lookup"] = daily["date"] - pd.Timedelta(days=1)
    daily = pd.merge_asof(daily.sort_values("gas_lookup"), gas.rename(columns={"date": "gas_date"}),
                          left_on="gas_lookup", right_on="gas_date", direction="backward",
                          tolerance=pd.Timedelta(days=7)).drop(columns=["gas_lookup", "gas_date"])
    daily = daily.sort_values("date").reset_index(drop=True)
    daily.to_csv(CACHE, index=False)
    print("Built and cached:", daily.shape)

daily.tail()

In [ ]:
# Cell 5: data checks
print("Missing values per column:\n", daily.isna().sum(), "\n")
print(daily.describe().round(2), "\n")

# Answer key from the Grid Status Block Prices panel
checks = {("2026-11-11", "da_on"): 72.15, ("2026-11-11", "rt_on"): 91.30, ("2026-11-12", "da_on"): 88.27}
for (day, col), expected in checks.items():
    row = daily.loc[daily["date"] == day, col]
    if len(row) and pd.notna(row.iloc[0]):
        print(f"{day} {col}: mine {row.iloc[0]:.2f} vs Grid Status {expected:.2f}")
    else:
        print(f"{day} {col}: not in data yet")

In [ ]:
# Cell 6: model frame (on-peak days only)
df = daily.dropna(subset=["da_on", "hdd", "gas"]).copy()
df = df.sort_values("date").reset_index(drop=True)
df["dow"] = df["date"].dt.dayofweek
df["month"] = df["date"].dt.month
df["year"] = df["date"].dt.year
df["heat_rate"] = df["da_on"] / df["gas"]
df["lag1"] = df["da_on"].shift(1)                                   # previous on-peak day's DA (known when bidding)
df["last_year"] = df["date"].map(df.set_index("date")["da_on"].rename(lambda d: d + pd.Timedelta(days=364)))
df = df.dropna(subset=["lag1"]).reset_index(drop=True)

train = df[df["date"] < TEST_START].copy()
test = df[df["date"] >= TEST_START].copy()
print(f"Train: {len(train)} days ({train.date.min():%Y-%m-%d} to {train.date.max():%Y-%m-%d})")
print(f"Test:  {len(test)} days ({test.date.min():%Y-%m-%d} to {test.date.max():%Y-%m-%d})")
print("\nCorrelation of drivers (watch for multicollinearity):")
print(df[["hdd", "cdd", "gas", "da_on"]].corr().round(2))

In [ ]:
# Cell 7: run the models
preds = pd.DataFrame(index=df.index)
y = df["da_on"]

# Benchmarks
preds["Naive: previous day"] = df["lag1"]
preds["Naive: same day last year"] = df["last_year"]

# Moving averages of past prices (shifted so only past data is used)
for w in [2, 5, 10, 20, 60]:
    preds[f"Moving average {w}d"] = y.shift(1).rolling(w).mean()
for w in [5, 20]:
    weights = np.arange(1, w + 1)
    preds[f"Weighted MA {w}d"] = y.shift(1).rolling(w).apply(lambda x: np.dot(x, weights) / weights.sum(), raw=True)

# Exponential smoothing: choose alpha on the training period only
tr = df["date"] < TEST_START
best_alpha = min(np.arange(0.05, 1.0, 0.05),
                 key=lambda a: (y - y.ewm(alpha=a, adjust=False).mean().shift(1))[tr].abs().mean())
preds[f"Exp. smoothing (alpha={best_alpha:.2f})"] = y.ewm(alpha=best_alpha, adjust=False).mean().shift(1)

# Regressions: fit on train, predict test
formulas = {
    "Regression: weather + gas + FE": "da_on ~ hdd + cdd + gas + C(dow) + C(month)",
    "Regression: + squared weather": "da_on ~ hdd + I(hdd**2) + cdd + I(cdd**2) + gas + C(dow) + C(month)",
    "Regression: + squared weather + lag": "da_on ~ hdd + I(hdd**2) + cdd + I(cdd**2) + gas + lag1 + C(dow) + C(month)",
}
fits = {}
for name, f in formulas.items():
    fits[name] = smf.ols(f, data=train).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    preds.loc[test.index, name] = fits[name].predict(test)

# Heat-rate model: Y = price / gas, then multiply back by gas
hr_fit = smf.ols("heat_rate ~ hdd + I(hdd**2) + cdd + I(cdd**2) + C(dow) + C(month)",
                 data=train).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
preds.loc[test.index, "Heat-rate model x gas"] = hr_fit.predict(test) * test["gas"]

# SARIMAX: ARMA(1,1) errors with weather, gas and weekday dummies as inputs
try:
    def exog(d):
        X = d[["hdd", "cdd", "gas"]].copy()
        X["hdd2"] = d["hdd"] ** 2
        for k in range(1, 5):
            X[f"dow{k}"] = (d["dow"] == k).astype(float)
        return X.astype(float).values
    sar_train = SARIMAX(train["da_on"].values, exog=exog(train), order=(1, 0, 1), trend="c").fit(disp=False)
    sar_all = SARIMAX(df["da_on"].values, exog=exog(df), order=(1, 0, 1), trend="c").filter(sar_train.params)
    one_step = sar_all.predict()                         # one-step-ahead, parameters estimated on train only
    preds.loc[test.index, "SARIMAX(1,0,1) + weather/gas"] = one_step[test.index]
except Exception as e:
    print("SARIMAX skipped:", e)

print("Models run:", len(preds.columns))

In [ ]:
# Cell 8: comparison table (test period only, same days for every model)
P = preds.loc[test.index].dropna()
A = y.loc[P.index]
winter = df.loc[P.index, "month"].isin([12, 1, 2])
rows = []
for m in P.columns:
    e = A - P[m]
    rows.append({"Model": m, "MAE": e.abs().mean(), "RMSE": np.sqrt((e ** 2).mean()),
                 "Std error": e.std(), "Bias (actual - pred)": e.mean(),
                 "MAE winter": e[winter].abs().mean(), "MAE rest": e[~winter].abs().mean(),
                 "Worst miss": e.abs().max()})
results = pd.DataFrame(rows).sort_values("MAE").reset_index(drop=True).round(2)
print(f"Out-of-sample comparison, {len(P)} on-peak days in {TEST_START[:4]} ($/MWh)\n")
try:
    print(results.to_markdown(index=False))
except Exception:
    print(results.to_string(index=False))
results.to_csv("model_comparison.csv", index=False)

In [ ]:
# Cell 9: sensitivities (what a trader cares about)
base = fits["Regression: weather + gas + FE"]
sens = pd.DataFrame({"Coefficient": base.params, "HAC std error": base.bse, "p-value": base.pvalues}).loc[["hdd", "cdd", "gas"]]
print("Training-period sensitivities, $/MWh on-peak:")
print(" hdd = per extra heating degree day | cdd = per extra cooling degree day | gas = per $1/MMBtu\n")
print(sens.round(3), "\n")

# Has weather sensitivity changed year to year? (coal retirements, data center load)
by_year = []
for yr, g in df.groupby("year"):
    if len(g) < 120:
        continue
    f = smf.ols("da_on ~ hdd + cdd + gas + C(dow)", data=g).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    by_year.append({"Year": yr, "Days": len(g), "Avg price": g["da_on"].mean(), "Avg gas": g["gas"].mean(),
                    "$/MWh per HDD": f.params["hdd"], "$/MWh per CDD": f.params["cdd"],
                    "Avg heat rate": g["heat_rate"].mean(), "Price std dev": g["da_on"].std()})
by_year = pd.DataFrame(by_year).round(2)
print("By year:")
try:
    print(by_year.to_markdown(index=False))
except Exception:
    print(by_year.to_string(index=False))

In [ ]:
# Cell 10: outliers to investigate (largest misses, all years)
full_fit = smf.ols(formulas["Regression: + squared weather"], data=df).fit()
df["fitted"] = full_fit.fittedvalues
df["residual"] = df["da_on"] - df["fitted"]
out = (df.reindex(df["residual"].abs().sort_values(ascending=False).index)
         .head(15)[["date", "da_on", "fitted", "residual", "temp", "hdd", "cdd", "gas"]].round(1))
out["date"] = out["date"].dt.strftime("%Y-%m-%d (%a)")
print("15 largest misses: look up what happened on each date (cold snap, outage, congestion, holiday)\n")
try:
    print(out.to_markdown(index=False))
except Exception:
    print(out.to_string(index=False))
out.to_csv("outliers.csv", index=False)

In [ ]:
# Cell 11: charts
best = results.loc[0, "Model"]
fig, axes = plt.subplots(2, 1, figsize=(11, 8))
ax = axes[0]
ax.plot(df.loc[P.index, "date"], A, label="Actual DA on-peak", color="black", lw=1.2)
ax.plot(df.loc[P.index, "date"], P[best], label=f"Best: {best}", lw=1)
ax.plot(df.loc[P.index, "date"], P["Naive: previous day"], label="Naive: previous day", lw=0.8, alpha=0.6)
ax.set_title(f"Western Hub on-peak DA: actual vs forecast ({TEST_START[:4]}, out-of-sample)")
ax.set_ylabel("$/MWh"); ax.legend()
ax = axes[1]
sc = ax.scatter(df["temp"], df["da_on"], c=df["gas"], cmap="viridis", s=8)
ax.set_xlabel("PJM average temperature (°F)"); ax.set_ylabel("On-peak DA ($/MWh)")
ax.set_title("Price vs temperature (color = Henry Hub $/MMBtu): the U-shape is heating and cooling demand")
plt.colorbar(sc, ax=ax, label="Gas $/MMBtu")
plt.tight_layout()
plt.savefig("pjm_results.png", dpi=150)
plt.show()

In [ ]:
# Cell 12: one-paragraph summary to paste back to Claude
print(f"""SUMMARY
Data: {df.date.min():%Y-%m-%d} to {df.date.max():%Y-%m-%d}, {len(df)} on-peak days; train {len(train)}, test {len(P)}.
Best model: {best} (MAE {results.loc[0, 'MAE']} $/MWh).
Naive previous-day MAE: {results.set_index('Model').loc['Naive: previous day', 'MAE']} $/MWh.
Sensitivity (train): {sens.loc['hdd', 'Coefficient']:.2f} $/MWh per HDD, {sens.loc['cdd', 'Coefficient']:.2f} per CDD, {sens.loc['gas', 'Coefficient']:.2f} per $1 gas.
Files saved: pjm_daily.csv, model_comparison.csv, outliers.csv, pjm_results.png
""")